# Giving Agents Tools and Knowledge: MCP, Skills and OKF

The full code of [Giving Agents Tools and Knowledge: MCP, Skills and OKF](https://residual-stream.lvtan71.workers.dev/posts/mcp-skills-and-okf/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/mcp-skills-and-okf) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/mcp-skills-and-okf.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.
The post's run read the blog's working tree with drafts in it; this snapshot is the last commit before, so the search scores differ in the last digits.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 0 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "mcp==2.2.*" "transformers==5.17.*" huggingface_hub pyyaml

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "mcp-skills-and-okf"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/mcp-skills-and-okf


In [2]:
!tar xzf blog-deb8375.tar.gz

## `mcp_client.py`

A host's side through the SDK: launches `blog_server.py`, lists its tools and resource template, calls both tools and reads a resource.

In [3]:
!{sys.executable} mcp_client.py

tool search_posts: Find the blog posts most relevant to a query (BM25 over the full text).
  input schema: {"type": "object", "properties": {"query": {"title": "Query", "type": "string"}, "k": {"default": 3, "title": "K", "type": "integer"}}, "required": ["query"], "...
tool read_post: Read the start of one post, by slug.
  input schema: {"type": "object", "properties": {"slug": {"title": "Slug", "type": "string"}, "max_words": {"default": 400, "title": "Max Words", "type": "integer"}}, "require...
protocol version: 2026-07-28


search_posts -> {'result': [{'slug': 'paged-attention-explained', 'title': 'Paged Attention: Managing the KV Cache Like Virtual Memory', 'score': 5.28}, {'slug': 'kv-cache-explained', 'title': 'The KV Cache: How Transformers Avoid Recomputing the Past', 'score': 4.98}, {'slug': 'flashattention-explained', 'title': 'FlashAttention: Exact Attention at the Speed of Memory', 'score': 4.45}]}
read_post -> [Post 20](/posts/kv-cache-explained/) sized the KV cache, and posts 18 and 23 showed the architectural ways to shrink it: share key-value heads (GQA), or store a
resource template post://{slug}: A post's full text, for a host to attach as context.
resources/read -> [Post 13](/posts/residual-stream-explained/) found two thirds of a Transformer b ...


The post printed:

```text
tool search_posts: Find the blog posts most relevant to a query (BM25 over the full text).
  input schema: {"type": "object", "properties": {"query": {"title": "Query", "type": "string"}, "k": {"default": 3, "title": "K", "type": "integer"}}, "required": ["query"], "...
tool read_post: Read the start of one post, by slug.
  input schema: {"type": "object", "properties": {"slug": {"title": "Slug", "type": "string"}, "max_words": {"default": 400, "title": "Max Words", "type": "integer"}}, "require...
protocol version: 2026-07-28
search_posts -> {'result': [{'slug': 'paged-attention-explained', 'title': 'Paged Attention: Managing the KV Cache Like Virtual Memory', 'score': 5.75}, {'slug': 'kv-cache-explained', 'title': 'The KV Cache: How Transformers Avoid Recomputing the Past', 'score': 5.44}, {'slug': 'flashattention-explained', 'title': 'FlashAttention: Exact Attention at the Speed of Memory', 'score': 4.88}]}
read_post -> [Post 20](/posts/kv-cache-explained/) sized the KV cache, and posts 18 and 23 showed the architectural ways to shrink it: share key-value heads (GQA), or store a
resource template post://{slug}: A post's full text, for a host to attach as context.
resources/read -> [Post 13](/posts/residual-stream-explained/) found two thirds of a Transformer b ...
```

## `wire.py`

The same tool call without an SDK: one JSON-RPC request written to the server's stdin, one reply read from its stdout.

In [4]:
!{sys.executable} wire.py

-> {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search_posts", "arguments": {"query": "speculative decoding draft", "k": 2}, "_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}, "io.modelcontextprotocol/clientInfo": {"name": "wire-demo", "version": "0.1"}}}}


<- {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"text": "{\n  \"slug\": \"speculative-decoding\",\n  \"title\": \"Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE\",\n  \"score\": 10.42\n}", "type": "text"}, {"text": "{\n  \"slug\": \"tensorrt-llm-explained\",\n  \"title\": \"TensorRT-LLM: Compiling Inference Engines Ahead of Time\",\n  \"score\": 9.87\n}", "type": "text"}], "isError": false, "resultType": "complete", "structuredContent": {"result": [{"slug": "speculative-decoding", "title": "Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE", "score": 10.42}, {"slug": "tensorrt-llm-explained", "title": "TensorRT-LLM: Compiling Inference Engines


The post printed:

```text
-> {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search_posts", "arguments": {"query": "speculative decoding draft", "k": 2}, "_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}, "io.modelcontextprotocol/clientInfo": {"name": "wire-demo", "version": "0.1"}}}}
<- {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"text": "{\n  \"slug\": \"speculative-decoding\",\n  \"title\": \"Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE\",\n  \"score\": 10.61\n}", "type": "text"}, {"text": "{\n  \"slug\": \"tensorrt-llm-explained\",\n  \"title\": \"TensorRT-LLM: Compiling Inference Engines Ahead of Time\",\n  \"score\": 10.05\n}", "type": "text"}], "isError": false, "resultType": "complete", "structuredContent": {"result": [{"slug": "speculative-decoding", "title": "Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE", "score": 10.61}, {"slug": "tensorrt-llm-explained", "title": "TensorRT-LLM: Compiling Inference Engine
```

## `tool_cost.py`

Tokens that tool definitions and skill metadata take in every prompt, counted with Qwen2.5's tokenizer.

In [5]:
!{sys.executable} tool_cost.py

this blog's 2 tools: 295 tokens of definitions, 148 per tool
  of which output schemas: 101 tokens
5 servers x 15 tools at that size: 11,062 tokens in every request's prompt

one skill's metadata (name and description): 41 tokens
  10 skills: metadata always loaded     410 tokens; every body loaded up front at ~4,000 tokens each    40,000 tokens
  50 skills: metadata always loaded   2,050 tokens; every body loaded up front at ~4,000 tokens each   200,000 tokens
 200 skills: metadata always loaded   8,200 tokens; every body loaded up front at ~4,000 tokens each   800,000 tokens


The post printed:

```text
this blog's 2 tools: 295 tokens of definitions, 148 per tool
  of which output schemas: 101 tokens
5 servers x 15 tools at that size: 11,062 tokens in every request's prompt

one skill's metadata (name and description): 41 tokens
  10 skills: metadata always loaded     410 tokens; every body loaded up front at ~4,000 tokens each    40,000 tokens
  50 skills: metadata always loaded   2,050 tokens; every body loaded up front at ~4,000 tokens each   200,000 tokens
 200 skills: metadata always loaded   8,200 tokens; every body loaded up front at ~4,000 tokens each   800,000 tokens
```

## `okf.py`

Writes a small Open Knowledge Format bundle, `bundle/`, from the front matter of a few posts, and prints its index and one concept.

In [6]:
!{sys.executable} okf.py

---
okf_version: "0.2"
---

# Residual Stream concepts

- [The KV Cache: How Transformers Avoid Recomputing the Past](concepts/kv-cache.md)
- [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](concepts/speculative-decoding.md)
- [Vector Search: Exact Search and the Approximate Indexes](concepts/hnsw.md)

---
type: Concept
title: "The KV Cache: How Transformers Avoid Recomputing the Past"
description: "Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to linear, worked through with colored matrices step by step — plus the memory cost formula and what it mea..."
tags: [llm, inference]
sources:
  - resource: https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/
    title: "The KV Cache: How Transformers Avoid Recomputing the Past"
generated:
  by: process:make_okf
  at: "2026-10-01"
status: draft
---

# Summary

Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to li

The post printed:

```text
---
okf_version: "0.2"
---

# Residual Stream concepts

- [The KV Cache: How Transformers Avoid Recomputing the Past](concepts/kv-cache.md)
- [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](concepts/speculative-decoding.md)
- [Vector Search: Exact Search and the Approximate Indexes](concepts/hnsw.md)

---
type: Concept
title: "The KV Cache: How Transformers Avoid Recomputing the Past"
description: "Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to linear, worked through with colored matrices step by step — plus the memory cost formula and what it mea..."
tags: [llm, inference]
sources:
  - resource: https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/
    title: "The KV Cache: How Transformers Avoid Recomputing the Past"
generated:
  by: process:make_okf
  at: "2026-10-01"
status: draft
---

# Summary

Why autoregressive decoding wastes work, how caching keys and values turns it from quadratic to linear, worked through with colored matrices step by step — plus the memory cost formula and what it means for real models.

# See also

[the full post](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/)
```